In [ ]:
import geopandas as gpd

# WSL 路径
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

# 筛选 LIA >= 10 km²
lia_large = lia[lia['LIAArea'] >= 30].copy()

# 提取关联的 RGI ID
all_rgi_ids = set()
for rgi_str in lia_large['AllRGIs'].dropna():
    ids = [x.strip() for x in str(rgi_str).split(';')]
    all_rgi_ids.update(ids)

# 筛选对应的现代冰川
rgi_matched = rgi[rgi['RGIId'].isin(all_rgi_ids)].copy()

print(f"LIA >= 10 km²: {len(lia_large)} 个")
print(f"对应现代冰川: {len(rgi_matched)} 个")

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'
from oggm import cfg, workflow, tasks, utils
WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_himalaya_great"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"

MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"

# 转换为 WGS84
lia_large = lia_large.to_crs("EPSG:4326")
rgi_matched = rgi_matched.to_crs("EPSG:4326")

print(f"LIA 坐标系: {lia_large.crs}")
print(f"RGI 坐标系: {rgi_matched.crs}")

dfs = []
for path in MB_PATHS:
    print(f"  读取: {path}")
    df = pd.read_csv(path)
    dfs.append(df)

mb_all = pd.concat(dfs, ignore_index=True)
print(f"  合并后总记录: {len(mb_all)}")

mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
print(f"  时段 {MB_PERIOD} 记录数: {len(mb_data)}")

mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')

rgi = rgi_matched

rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
print(f"清理后冰川数: {len(rgi)}")

rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print(f"有MB数据的冰川: {len(rgi)}")

In [ ]:
cfg.initialize(logging_level='WARNING')

cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH

cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 16

gdirs = workflow.init_glacier_directories(rgi, reset=False)

In [ ]:


# cfg.PARAMS['baseline_climate'] = 'CRU'
#
# for gdir in gdirs:
#     tasks.process_cru_data(gdir)

In [ ]:
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['use_intersects'] = False
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['use_multiple_flowlines'] = True
cfg.PARAMS['use_multiprocessing'] = True

In [ ]:
for gdir in gdirs:
    tasks.define_glacier_region(gdir, source='USER')

# from tqdm import tqdm
# failed = []
# for gdir in tqdm(gdirs):
#     try:
#         tasks.define_glacier_region(gdir, source='USER')
#     except Exception as e:
#         failed.append((gdir.rgi_id, str(e)))
#
# print(f"失败: {len(failed)}")
# for rgi_id, err in failed[:5]:
#     print(f"  {rgi_id}: {err}") # 多线程定义冰川区域似会出错

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True
execute_entity_task(tasks.glacier_masks, gdirs);
execute_entity_task(tasks.compute_centerlines, gdirs)#中流线使用多线程会报错
execute_entity_task(tasks.initialize_flowlines, gdirs)
execute_entity_task(tasks.compute_downstream_line, gdirs)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs)
execute_entity_task(tasks.catchment_area, gdirs)
execute_entity_task(tasks.catchment_intersections, gdirs)
execute_entity_task(tasks.catchment_width_geom, gdirs)
execute_entity_task(tasks.catchment_width_correction, gdirs)

In [ ]:
cfg.PARAMS['baseline_climate'] = 'W5E5'  # 清空，使用默认

for gdir in gdirs:
    tasks.process_climate_data(gdir)

In [ ]:
# MB 校准
print("执行 MB 校准...")
for gdir in gdirs:
    try:
        rgi_id = gdir.rgi_id
        if rgi_id not in mb_data.index:
            continue
        ref_mb = mb_data.loc[rgi_id, 'dmdtda_kg']
        tasks.mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            ref_mb_years=(2000, 2020),
            overwrite_gdir=True,
            calibrate_param1='melt_f',
            calibrate_param2='temp_bias',
        )
    except Exception as e:
        print(f"  {rgi_id} 校准失败: {e}")

# 反演任务
print("执行反演任务...")
execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs)
execute_entity_task(tasks.prepare_for_inversion, gdirs)
execute_entity_task(tasks.mass_conservation_inversion, gdirs)
execute_entity_task(tasks.filter_inversion_output, gdirs)
execute_entity_task(tasks.distribute_thickness_per_altitude, gdirs)

print("完成!")

In [ ]:
failed_ids = [
    'RGI60-15.01144', 'RGI60-15.01154', 'RGI60-15.03709',
    'RGI60-15.05468', 'RGI60-15.05534', 'RGI60-15.06919',
    'RGI60-15.07103', 'RGI60-15.07104', 'RGI60-15.07125',
    'RGI60-15.08955'
]

# 从 rgi 和 mb_data 中查询
failed_info = rgi[rgi['RGIId'].isin(failed_ids)][['RGIId', 'Area']].copy()
failed_info = failed_info.merge(
    mb_data[['dmdtda', 'dmdtda_kg']].reset_index(),
    left_on='RGIId',
    right_on='rgiid',
    how='left'
)
print(failed_info[['RGIId', 'Area', 'dmdtda', 'dmdtda_kg']].to_string())

In [ ]:
failed_ids = [
    'RGI60-15.01144', 'RGI60-15.01154', 'RGI60-15.03709',
    'RGI60-15.05468', 'RGI60-15.05534', 'RGI60-15.06919',
    'RGI60-15.07103', 'RGI60-15.07104', 'RGI60-15.07125',
    'RGI60-15.08955'
]

# 从 gdirs 中剔除
gdirs_clean = [gdir for gdir in gdirs if gdir.rgi_id not in failed_ids]

print(f"剔除前: {len(gdirs)}, 剔除后: {len(gdirs_clean)}")

In [ ]:
import xarray as xr
output_dir = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/bare_dem"
os.makedirs(output_dir, exist_ok=True)

for gdir in gdirs_clean:
    try:
        with xr.open_dataset(gdir.get_filepath('gridded_data')) as ds:
            bare = ds['topo'] - ds['distributed_thickness']
            bare.rio.to_raster(f"{output_dir}/{gdir.rgi_id}_bare.tif")
    except:
        pass

print("完成!")

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.crs import CRS
from rasterio.windows import Window
import numpy as np
import xarray as xr
from tqdm import tqdm

dem_path = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
output_path = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/Himalaya_bare_DEM.tif"

# 复制原始 DEM
with rasterio.open(dem_path) as src:
    dem = src.read(1).astype(np.float32)
    profile = src.profile.copy()
    dem_transform = src.transform
    dem_crs = src.crs
    dem_bounds = src.bounds

profile.update(dtype=np.float32)

# 处理每个冰川
for gdir in tqdm(gdirs_clean):
    try:
        with xr.open_dataset(gdir.get_filepath('gridded_data')) as ds:
            thick = np.nan_to_num(ds['distributed_thickness'].values.astype(np.float32), nan=0.0)
            x, y = ds.x.values, ds.y.values

        res = x[1] - x[0]
        glacier_transform = rasterio.transform.from_origin(x[0]-res/2, y[0]+res/2, res, res)
        glacier_crs = CRS.from_proj4(gdir.grid.proj.to_proj4())

        # 冰川边界转经纬度
        gb = transform_bounds(glacier_crs, dem_crs, x[0]-res/2, y[-1]-res/2, x[-1]+res/2, y[0]+res/2)

        # 计算窗口
        col_off = int((gb[0] - dem_bounds.left) / dem_transform[0])
        row_off = int((dem_bounds.top - gb[3]) / (-dem_transform[4]))
        width = int((gb[2] - gb[0]) / dem_transform[0]) + 1
        height = int((gb[3] - gb[1]) / (-dem_transform[4])) + 1

        # 边界检查
        col_off = max(0, col_off)
        row_off = max(0, row_off)
        width = min(width, dem.shape[1] - col_off)
        height = min(height, dem.shape[0] - row_off)

        # 窗口 transform
        clip_transform = rasterio.transform.from_origin(
            dem_bounds.left + col_off * dem_transform[0],
            dem_bounds.top - row_off * (-dem_transform[4]),
            dem_transform[0], -dem_transform[4]
        )

        # 重投影冰厚
        thick_reproj = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=thick,
            destination=thick_reproj,
            src_transform=glacier_transform,
            src_crs=glacier_crs,
            dst_transform=clip_transform,
            dst_crs=dem_crs,
            resampling=Resampling.bilinear
        )

        # 减去冰厚
        dem[row_off:row_off+height, col_off:col_off+width] -= thick_reproj

    except Exception as e:
        print(f"{gdir.rgi_id}: {e}")

# 保存
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(dem, 1)

print(f"完成: {output_path}")

In [ ]:
# 只检查文件元信息，不读数据
import rasterio
import xarray as xr
from rasterio import CRS
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import Window
output_path = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/Himalaya_bare_DEM.tif"
dem_path = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
with rasterio.open(output_path) as src:
    print(f"输出: {src.shape}, CRS: {src.crs}")
    dem_transform = src.transform
    dem_crs = src.crs
    dem_bounds = src.bounds
print(f"输出: {src.shape}, CRS: {src.crs}")

# 采样几个冰川区域验证
import random
sample_gdirs = random.sample(gdirs_clean[:100], 3)

for gdir in sample_gdirs:
    with xr.open_dataset(gdir.get_filepath('gridded_data')) as ds:
        if 'distributed_thickness' not in ds:
            continue
        x, y = ds.x.values, ds.y.values

    res = x[1] - x[0]
    glacier_crs = CRS.from_proj4(gdir.grid.proj.to_proj4())
    gb = transform_bounds(glacier_crs, dem_crs, x[0]-res/2, y[-1]-res/2, x[-1]+res/2, y[0]+res/2)

    col_off = int((gb[0] - dem_bounds.left) / dem_transform[0])
    row_off = int((dem_bounds.top - gb[3]) / (-dem_transform[4]))
    width = min(500, int((gb[2] - gb[0]) / dem_transform[0]))
    height = min(500, int((gb[3] - gb[1]) / (-dem_transform[4])))

    window = Window(col_off, row_off, width, height)

    with rasterio.open(dem_path) as src:
        orig = src.read(1, window=window)
    with rasterio.open(output_path) as src:
        bare = src.read(1, window=window)

    diff = orig - bare
    print(f"{gdir.rgi_id}: 差值像素 {(diff>0).sum()}, 最大 {diff.max():.1f}m")

In [ ]:
# 合并所有失败的 ID
failed_ids = [
    'RGI60-15.01144', 'RGI60-15.01154', 'RGI60-15.03709',
    'RGI60-15.05468', 'RGI60-15.05534', 'RGI60-15.06919',
    'RGI60-15.07103', 'RGI60-15.07104', 'RGI60-15.07125',
    'RGI60-15.08955'
]

no_thick_ids = [
    'RGI60-14.17995', 'RGI60-14.17997', 'RGI60-14.17999',
    'RGI60-14.18007', 'RGI60-14.18008', 'RGI60-14.18034',
    'RGI60-14.18035', 'RGI60-14.18036', 'RGI60-14.18037',
    'RGI60-14.18038', 'RGI60-14.18040', 'RGI60-14.18041',
    'RGI60-14.18042', 'RGI60-14.18044', 'RGI60-14.18045'
]

all_failed = failed_ids + no_thick_ids

# 排除 RGI
rgi_clean = rgi[~rgi['RGIId'].isin(all_failed)].copy()
print(f"RGI 原始: {len(rgi)}, 排除后: {len(rgi_clean)}")

# 排除 LIA
def has_failed_rgi(all_rgis_str):
    if pd.isna(all_rgis_str):
        return False
    ids = [x.strip() for x in str(all_rgis_str).split(';')]
    return any(rgi_id in all_failed for rgi_id in ids)

lia_clean = lia_large[~lia_large['AllRGIs'].apply(has_failed_rgi)].copy()
print(f"LIA 原始: {len(lia_large)}, 排除后: {len(lia_clean)}")

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString
from tqdm import tqdm

# 排除失败的gdirs
all_failed = failed_ids + no_thick_ids
gdirs_clean = [gdir for gdir in gdirs if gdir.rgi_id not in all_failed]

# 提取flowlines
flowline_records = []
for gdir in tqdm(gdirs_clean, desc="提取flowlines"):
    try:
        fls = gdir.read_pickle('inversion_flowlines')
        for i, fl in enumerate(fls):
            # 用 fl.line.xy
            try:
                x, y = fl.line.xy
            except AttributeError:
                continue

            # 转换为真实坐标（lon, lat）
            coords = []
            for xi, yi in zip(x, y):
                lon, lat = gdir.grid.ij_to_crs(xi, yi, crs='EPSG:4326')
                coords.append((lon, lat))

            if len(coords) >= 2:
                line_geom = LineString(coords)
                flowline_records.append({
                    'rgi_id': gdir.rgi_id,
                    'fl_id': i,
                    'geometry': line_geom,
                    # 用getattr安全获取，没有就用几何长度
                    'length_m': getattr(fl, 'length_m', None) or line_geom.length * 111000,  # 粗略转换
                })
    except Exception as e:
        print(f"{gdir.rgi_id}: {e}")

print(f"提取了 {len(flowline_records)} 条flowlines")

# 创建GeoDataFrame
flowlines_gdf = gpd.GeoDataFrame(flowline_records, geometry='geometry', crs="EPSG:4326")

# 导出
output_dir = "/mnt/f/oggm_related/oggm_LIA_himalaya_great"
flowlines_gdf.to_file(f"{output_dir}/flowlines_clean.gpkg", driver="GPKG")
lia_clean.to_file(f"{output_dir}/lia_clean.gpkg", driver="GPKG")
rgi_clean.to_file(f"{output_dir}/rgi_clean.gpkg", driver="GPKG")

print(f"导出完成:")
print(f"  flowlines: {len(flowlines_gdf)} 条, 冰川数: {flowlines_gdf['rgi_id'].nunique()}")
print(f"  LIA: {len(lia_clean)} 个")
print(f"  RGI: {len(rgi_clean)} 个")

In [ ]:
# 导出
output_dir = "/mnt/f/oggm_related/oggm_LIA_himalaya_great"

# 导出为shapefile
flowlines_gdf.to_file(f"{output_dir}/flowlines_clean.shp", driver="ESRI Shapefile")
lia_clean.to_file(f"{output_dir}/lia_clean.shp", driver="ESRI Shapefile")
rgi_clean.to_file(f"{output_dir}/rgi_clean.shp", driver="ESRI Shapefile")

print(f"导出完成:")
print(f"  flowlines: {len(flowlines_gdf)} 条, 冰川数: {flowlines_gdf['rgi_id'].nunique()}")
print(f"  LIA: {len(lia_clean)} 个")
print(f"  RGI: {len(rgi_clean)} 个")